# Spicerack - Recipe Recommender
### DS Club Project - Spring 2026

The idea is pretty simple: you tell us what spices you have, we tell you what you can cook.
Also tells you what spice to buy next to unlock the most new recipes.

Dataset: RecipeNLG (~2.2M recipes from Kaggle)

GitHub: https://github.com/laniel123/DS-Club-Project-Showcase-SpiceRack

---
## Step 1 - Change your inputs here

This is the only cell you need to touch. Edit your pantry and must-use lists, then run all cells.

If you put something in must-use that isnt in your pantry, it gets added automatically.
If a spice isnt recognized it just gets skipped with a warning.

In [9]:
# change these two lists and run everything

# spices you actually have right now
MY_PANTRY = [
    "garlic",
    "cumin",
    "paprika",
    "chili powder",
    "oregano",
    "black pepper",
    "salt",
    "cinnamon",
    "ginger",
]

# spices that MUST show up in every recipe we recommend
# leave as [] if you dont care
MUST_USE = [
    "cumin",
    "garlic",
]

# path to your csv file
CSV_PATH = "cookingdataset/RecipeNLG_dataset.csv"

# how many recipes to load - lower this if its running slow
SAMPLE_SIZE = None  # set to None to load all, or a number like 10000 for a sample

---
## Step 2 - Imports and setup
Run this once, dont touch it

In [12]:
import re
import warnings
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.decomposition import NMF
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import pairwise_distances

# pulling all our spice data from the external file
# spice_data.py needs to be in the same folder as this notebook
from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES, FLAVOR_PROFILES, REGION_PROFILES

print(f"loaded {len(CANONICAL_SPICES)} spices, {len(FLAVOR_PROFILES)} flavor profiles, {len(REGION_PROFILES)} regions")

loaded 179 spices, 15 flavor profiles, 16 regions


In [ ]:
# helper functions used throughout

from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES

def clean(s) -> str:
    # lowercase and strip weird characters
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def to_canonical(spice):
    # normalize a spice name and apply alias mapping
    n = clean(spice)
    return clean(ALIASES.get(n, n))


def validate_spices(raw_list, label):
    # make sure everything the user typed is actually in our vocabulary
    # warns on anything we dont recognize and just skips it
    valid = set()
    bad = []
    for s in raw_list:
        c = to_canonical(s)
        if c in CANONICAL_SPICES:
            valid.add(c)
        else:
            bad.append(s)
    if bad:
        print(f"warning ({label}): didnt recognize these, skipping: {bad}")
    return valid


# build regex patterns for spice extraction from recipe text
# sorted by length so longer matches beat shorter ones (smoked paprika before paprika)
SPICE_PATTERNS = sorted(
    [(sp, clean(sp), re.compile(rf"(^| ){re.escape(clean(sp))}( |$)")) for sp in SPICES],
    key=lambda x: -len(x[0])
)

def get_spices_from_recipe(ingredients):
    # takes a recipe's ingredient list and returns which spices are in it
    if isinstance(ingredients, list):
        raw = " ".join(str(x) for x in ingredients)
    else:
        raw = str(ingredients)
    text = clean(raw)
    found = set()
    for _, norm, pat in SPICE_PATTERNS:
        if pat.search(" " + text + " "):
            found.add(norm)
    return {to_canonical(sp) for sp in found}


def parse_ingredient_string(x) -> list[str]:
    # ingredients are stored as a stringified python list in the csv
    # like '["1 cup flour", "2 eggs"]' so we parse it back
    if isinstance(x, list):
        return [str(i) for i in x]
    if not isinstance(x, str):
        return []
    s = x.strip()
    if s.startswith("[") and s.endswith("]"):
        items = re.findall(r"'([^']*)'|\"([^\"]*)\"", s)
        parsed = [a if a else b for a, b in items]
        return parsed if parsed else [s]
    return [s]


print("helper functions ready")

helper functions ready


---
## Step 3 - Load the data
Run this once per session. Takes a minute on the full dataset.

In [14]:
print(f"loading {CSV_PATH}...")

df = pd.read_csv(CSV_PATH)

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# parse ingredients column and extract which spices each recipe uses
df["ingredients_parsed"] = df["ingredients"].apply(parse_ingredient_string)
df["spices"] = df["ingredients_parsed"].apply(get_spices_from_recipe)

# binary matrix: rows = recipes, columns = spices, 1 if recipe contains that spice
mlb = MultiLabelBinarizer(classes=sorted(CANONICAL_SPICES))
X = np.asarray(mlb.fit_transform(df["spices"]))

print(f"done - {len(df):,} recipes loaded")
print(f"matrix shape: {X.shape}")
print(f"avg spices per recipe: {X.sum(axis=1).mean():.1f}")

loading cookingdataset/RecipeNLG_dataset.csv...
done - 2,231,142 recipes loaded
matrix shape: (2231142, 179)
avg spices per recipe: 2.0


In [ ]:
sample_df = df.sample(n=1000, random_state=42)


# save a smaller sample for quick testing without loading the whole thing every time

#sample_df.to_csv('sample_dataset.csv', index=False)

In [23]:
sample_df.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices'],
      dtype='object')

In [24]:
sample_df.head(15)

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices
2015528,2015528,Marinated Flank Steak Recipe,"[""1 1/2 pound flank steak"", ""1/2 c. finely min...","[""Remove tenderloin from steak."", ""Score meat....",cookeatshare.com/recipes/marinated-flank-steak...,Recipes1M,"[""flank steak"", ""green onions"", ""red wine"", ""s...","[1 1/2 pound flank steak, 1/2 c. finely minced...","{ginger, garlic, black pepper}"
1608734,1608734,French Chicken Stew,"[""1 tablespoon rosemary"", ""1 teaspoon thyme"", ...","[""combine all ingredients in slow cooker (6 qu...",www.yummly.com/recipe/French-Chicken-Stew-1433580,Gathered,"[""rosemary"", ""thyme"", ""bay leaves"", ""paprika"",...","[1 tablespoon rosemary, 1 teaspoon thyme, 3 ba...","{bay leaf, rosemary, paprika, thyme}"
778500,778500,Glazed Carrots,"[""3 to 4 carrots"", ""1 1/2 Tbsp. butter"", ""1/3 ...","[""Cook 3 to 4 carrots; cut crosswise in 1-inch...",www.cookbooks.com/Recipe-Details.aspx?id=1011892,Gathered,"[""carrots"", ""butter"", ""brown sugar"", ""lemon ri...","[3 to 4 carrots, 1 1/2 Tbsp. butter, 1/3 c. br...",{}
1334975,1334975,Moms Pie Dough,"[""4.5 Cups Flour"", ""1.5 Tsp Salt"", ""Pinch Baki...","[""Mix all dry ingredients in a bowl."", """", ""Ad...",www.epicurious.com/recipes/member/views/moms-p...,Gathered,"[""Flour"", ""Salt"", ""Baking Powder"", ""Sugar"", ""C...","[4.5 Cups Flour, 1.5 Tsp Salt, Pinch Baking Po...",{salt}
116562,116562,Pretzel Salad Or Dessert,"[""2 c. crushed small thin pretzels (sticks)"", ...","[""Mix and press in baking pan, approximately 1...",www.cookbooks.com/Recipe-Details.aspx?id=106723,Gathered,"[""thin pretzels"", ""margarine""]","[2 c. crushed small thin pretzels (sticks), 3/...",{}
1712896,1712896,Citrus Syrup,"[""3/4 cup sugar"", ""1/2 cup fresh orange juice""...","[""In a 1 1/2-quart saucepan stir together suga...",www.epicurious.com/recipes/food/views/citrus-s...,Recipes1M,"[""sugar"", ""orange juice"", ""lemon juice""]","[3/4 cup sugar, 1/2 cup fresh orange juice, 1/...",{}
1306450,1306450,Cranberry And Candied Orange Chutney,"[""1 large navel orange with skin"", ""7 cups wat...","[""Cut orange into 1/4-inch-thick rounds; cut r...",www.epicurious.com/recipes/food/views/cranberr...,Gathered,"[""orange with skin"", ""water"", ""sugar"", ""cinnam...","[1 large navel orange with skin, 7 cups water,...","{cinnamon, allspice}"
1345812,1345812,Tau Kua He Ci Medan'S Favourite Food,"[""1 slices Gravy ingredients (A) - onion"", ""3 ...","[""The condiments:"", ""- Large prawns, fried in ...",www.epicurious.com/recipes/member/views/tau-ku...,Gathered,"[""Gravy ingredients"", ""garlic"", ""Gravy ingredi...","[1 slices Gravy ingredients (A) - onion, 3 clo...","{cloves, garlic, salt}"
692271,692271,Jamaica Barbecue Sauce,"[""1 1/2 c. cider vinegar"", ""4 tsp. lemon juice...","[""Mix ingredients well."", ""Pour into jar."", ""K...",www.cookbooks.com/Recipe-Details.aspx?id=470060,Gathered,"[""cider vinegar"", ""lemon juice"", ""Worcestershi...","[1 1/2 c. cider vinegar, 4 tsp. lemon juice, 3...","{mustard, garlic, cayenne, salt}"
633422,633422,Dill Dip,"[""2/3 c. sour cream"", ""2/3 c. Hellmann's mayo""...","[""Mix all ingredients together and chill overn...",www.cookbooks.com/Recipe-Details.aspx?id=109385,Gathered,"[""sour cream"", ""mayo"", ""parsley flakes"", ""onio...","[2/3 c. sour cream, 2/3 c. Hellmann's mayo, 1 ...","{parsley, salt, dill, dried onion, onion flakes}"


---
## Step 4 - Model functions
These dont need to be touched either. Just run them.

In [ ]:
def get_flavor_profiles(pantry, min_coverage=0.3):
    # check how well the users spices cover each flavor profile
    # only returns profiles where they have at least min_coverage fraction of the spices
    results = []
    for profile, spice_set in FLAVOR_PROFILES.items():
        matched = pantry & spice_set
        coverage = len(matched) / len(spice_set)
        if coverage >= min_coverage:
            results.append((profile, round(coverage, 2), sorted(matched)))
    return sorted(results, key=lambda x: -x[1])


def get_regions(profile_names):
    # maps matched flavor profiles to culinary regions
    results = []
    for region, profiles in REGION_PROFILES.items():
        matched = [p for p in profiles if p in profile_names]
        if matched:
            results.append((region, matched))
    return sorted(results, key=lambda x: -len(x[1]))


def recommend(pantry, must_use, top_k=10, min_match=2):
    # jaccard similarity - compares users spice set to each recipes spice set
    # must_use filter removes any recipe that doesnt have all the required spices
    user_vec = np.asarray(mlb.transform([pantry]))

    if must_use:
        spice_cols = list(mlb.classes_)
        must_idx = [spice_cols.index(s) for s in must_use if s in spice_cols]
        if must_idx:
            must_mask = X[:, must_idx].min(axis=1).astype(bool)
        else:
            must_mask = np.ones(len(df), dtype=bool)
    else:
        must_mask = np.ones(len(df), dtype=bool)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sims = 1.0 - pairwise_distances(user_vec, X, metric="jaccard").flatten()

    match_counts = (X & user_vec).sum(axis=1)
    valid = np.where(must_mask & (match_counts >= min_match))[0]

    if len(valid) == 0:
        print("no recipes matched the must-use filter, showing best results without it")
        valid = np.where(match_counts >= min_match)[0]

    ranked = valid[np.lexsort((-match_counts[valid], -sims[valid]))][:top_k]

    out = df.loc[ranked, ["title"]].copy()
    out["similarity"]     = sims[ranked].round(3)
    out["matched_spices"] = df.loc[ranked, "spices"].apply(lambda s: sorted(s & pantry))
    out["num_matched"]    = match_counts[ranked]
    return out.sort_values(["similarity", "num_matched"], ascending=False).reset_index(drop=True)


def tag_region(recipe_spices):
    # assign a single region label to a recipe based on its spice set
    profiles = get_flavor_profiles(recipe_spices, min_coverage=0.2)
    if not profiles:
        return "Other"
    top_profile = profiles[0][0]
    for region, region_profiles in REGION_PROFILES.items():
        if top_profile in region_profiles:
            return region
    return "Other"


def suggest_next_spice(pantry, top_k=5, min_match=2, threshold=0.4):
    # for every spice you dont have, simulate adding it
    # count how many new recipes become available and rank by that
    baseline = recommend(pantry, must_use=set(), top_k=10_000, min_match=min_match)
    baseline_titles = set(baseline["title"])
    baseline_count  = len(baseline[baseline["similarity"] >= threshold])

    missing = [s for s in CANONICAL_SPICES if s not in pantry]
    print(f"baseline: {baseline_count} recipes above {threshold} similarity")
    print(f"testing {len(missing)} candidate spices...")

    results = []
    for spice in missing:
        expanded = pantry | {spice}
        new_recs = recommend(expanded, must_use=set(), top_k=10_000, min_match=min_match)
        new_count  = len(new_recs[new_recs["similarity"] >= threshold])
        new_titles = set(new_recs["title"]) - baseline_titles
        results.append({
            "spice":          spice,
            "newly_unlocked": new_count - baseline_count,
            "examples":       list(new_titles)[:3],
        })

    return pd.DataFrame(results).sort_values("newly_unlocked", ascending=False).head(top_k).reset_index(drop=True)


print("model functions ready")

model functions ready


---
## Step 5 - Results
Edit step 1 and re-run just this cell whenever you want to try different spices.

In [ ]:
# validate the inputs from step 1
pantry   = validate_spices(MY_PANTRY, label="pantry")
must_use = validate_spices(MUST_USE,  label="must-use")

extras = must_use - pantry
if extras:
    print(f"added to pantry automatically: {extras}")
    pantry |= extras

print(f"\npantry:   {sorted(pantry)}")
print(f"must-use: {sorted(must_use)}")


# --- flavor profiles ---
print("\n" + "-" * 55)
print("flavor profiles")
print("-" * 55)

profiles = get_flavor_profiles(pantry)
if not profiles:
    print("no strong profiles found - try adding more spices")
else:
    for name, score, matched in profiles:
        bar = "#" * int(score * 10) + "." * (10 - int(score * 10))
        print(f"  [{bar}] {int(score*100)}%  {name}")
        print(f"           spices: {', '.join(matched)}")


# --- culinary regions ---
print("\n" + "-" * 55)
print("cuisines you can cook")
print("-" * 55)

regions = get_regions([p[0] for p in profiles])
if not regions:
    print("no strong regional match found")
else:
    for region, matched_profiles in regions:
        print(f"  {region}")
        print(f"    via: {', '.join(matched_profiles)}")


# --- top recipe recommendations ---
print("\n" + "-" * 55)
if must_use:
    print(f"top recipes (must contain: {sorted(must_use)})")
else:
    print("top recipes")
print("-" * 55)

recs = recommend(pantry, must_use, top_k=10, min_match=2)
print(recs[["title", "similarity", "matched_spices", "num_matched"]].to_string(index=False))


# --- recipes grouped by region ---
print("\n" + "-" * 55)
print("recipes by region")
print("-" * 55)

pool = recommend(pantry, must_use, top_k=200, min_match=2)
pool["region"] = pool["title"].apply(
    lambda t: tag_region(
        df[df["title"] == t]["spices"].iloc[0]
        if (df["title"] == t).any() else set()
    )
)

for region in pool["region"].unique():
    if region == "Other":
        continue
    subset = pool[pool["region"] == region].head(3)
    print(f"\n  {region}")
    for _, row in subset.iterrows():
        print(f"    - {row['title']}  (similarity: {row['similarity']})")


# --- what spice should you buy next ---
print("\n" + "-" * 55)
print("spices to buy next")
print("-" * 55)

next_spice = suggest_next_spice(pantry, top_k=5, min_match=2)
for _, row in next_spice.iterrows():
    print(f"  {row['spice']:<25} {row['newly_unlocked']:+} new recipes")
    if row["examples"]:
        print(f"  {'':25} e.g. {row['examples'][0]}")


pantry:   ['black pepper', 'chili powder', 'cinnamon', 'cumin', 'garlic', 'ginger', 'oregano', 'paprika', 'salt']
must-use: ['cumin', 'garlic']

-------------------------------------------------------
flavor profiles
-------------------------------------------------------
  [###.......] 33%  American BBQ & Southern
           spices: black pepper, chili powder, cumin, garlic, paprika

-------------------------------------------------------
cuisines you can cook
-------------------------------------------------------
  American BBQ / Southern
    via: American BBQ & Southern

-------------------------------------------------------
top recipes (must contain: ['cumin', 'garlic'])
-------------------------------------------------------
                            title  similarity                                                                matched_spices  num_matched
                Fajitas Seasoning       0.889   [black pepper, chili powder, cumin, garlic, ginger, oregano, paprika, sa